# 🧠 La Neurona — Funciones de Activación
### Comparación: Identidad (sin activación) vs Tanh
---
Una neurona calcula `z = w·X + b` y luego aplica una **función de activación** `f(z)`.

| Función | Fórmula | Rango de salida | Uso típico |
|---|---|---|---|
| **Identidad** | `f(z) = z` | (−∞, +∞) | Regresión, capa de salida lineal |
| **Tanh** | `f(z) = (eᶻ − e⁻ᶻ) / (eᶻ + e⁻ᶻ)` | (−1, +1) | Clasificación, capas ocultas |

> 💡 Usar identidad es equivalente a **no aplicar ninguna función de activación**: la neurona queda totalmente lineal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')
print('Librerías cargadas ✓')

In [ ]:
# ── Paleta de colores ─────────────────────────────────────────────────────────
AZUL  = '#378ADD'
ROJO  = '#D85A30'
VERDE = '#3A9E6A'
NEGRO = '#1a1a1a'
NARAN = '#e85d3a'
MORA  = '#7B3FA0'
GRIS  = '#888888'

# ── Funciones de activación y sus derivadas ───────────────────────────────────
def f_identidad(z):  return z
def df_identidad(z): return np.ones_like(z)

def f_tanh(z):  return np.tanh(z)
def df_tanh(z): return 1.0 - np.tanh(z) ** 2

ACTIVACIONES = {
    'identidad': (f_identidad, df_identidad),
    'tanh':      (f_tanh,      df_tanh)
}

# ── Clase Neurona ─────────────────────────────────────────────────────────────
class Neurona:
    def __init__(self, n_features, activacion='identidad', seed=42):
        np.random.seed(seed)
        self.w  = np.random.randn(n_features) * 0.1
        self.b  = 0.0
        self.fn, self.dfn = ACTIVACIONES[activacion]
        self.nombre = activacion
        self.loss_hist   = []
        self.grad_w_hist = []   # historial de magnitud media del gradiente de w

    def forward(self, X):
        self.last_z = np.dot(X, self.w) + self.b
        return self.fn(self.last_z)

    def MSE(self, y_pred, y_true):
        return np.mean((y_pred - y_true) ** 2)

    def fit(self, X, y, lr=0.01, epochs=300):
        self.loss_hist   = []
        self.grad_w_hist = []
        N = X.shape[0]
        for _ in range(epochs):
            y_pred = self.forward(X)
            self.loss_hist.append(self.MSE(y_pred, y))
            # Regla de la cadena: dL/dw = (dL/dŷ) * f'(z) * x
            delta = (2 / N) * (y_pred - y) * self.dfn(self.last_z)
            dw    = np.dot(X.T, delta)
            db    = np.sum(delta)
            self.grad_w_hist.append(float(np.mean(np.abs(dw))))
            self.w -= lr * dw
            self.b -= lr * db

print('Funciones de activación y clase Neurona listos ✓')

---
## Módulo 1 — Visualización de las funciones de activación
Observa la forma de cada función y su derivada (el gradiente que viaja hacia atrás durante el entrenamiento).

In [ ]:
z = np.linspace(-4, 4, 400)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), facecolor='white')
fig.suptitle('Funciones de Activación y sus Derivadas', fontsize=13, fontweight='bold')

# ── Identidad ──
axes[0].plot(z, f_identidad(z),  color=AZUL, lw=2.5, label='f(z) = z')
axes[0].plot(z, df_identidad(z), color=AZUL, lw=1.8, ls='--', alpha=0.7, label="f'(z) = 1  [constante]")
axes[0].axhline(0, color='gray', lw=0.7, ls=':')
axes[0].axvline(0, color='gray', lw=0.7, ls=':')
axes[0].fill_between(z, df_identidad(z), alpha=0.08, color=AZUL)
axes[0].set_title('Identidad  (= sin activación)', fontsize=11)
axes[0].set_xlabel('z'); axes[0].set_ylabel('f(z)')
axes[0].set_ylim(-2.5, 2.5)
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.25)
axes[0].text(2.2, 1.15, "f'=1\nsiempre", fontsize=8, color=AZUL, ha='center')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# ── Tanh ──
axes[1].plot(z, f_tanh(z),  color=MORA, lw=2.5, label='f(z) = tanh(z)')
axes[1].plot(z, df_tanh(z), color=MORA, lw=1.8, ls='--', alpha=0.7, label="f'(z) = 1 − tanh²(z)")
axes[1].axhline( 1, color='gray', lw=0.6, ls=':', alpha=0.7)
axes[1].axhline(-1, color='gray', lw=0.6, ls=':', alpha=0.7)
axes[1].axhline( 0, color='gray', lw=0.7, ls=':')
axes[1].axvline( 0, color='gray', lw=0.7, ls=':')
axes[1].fill_between(z, df_tanh(z), alpha=0.08, color=MORA)
axes[1].set_title('Tanh  (saturación en ±1)', fontsize=11)
axes[1].set_xlabel('z'); axes[1].set_ylabel('f(z)')
axes[1].set_ylim(-1.5, 1.5)
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.25)
axes[1].annotate('Zona saturada\nf\'≈0', xy=(-3.5, -0.05), fontsize=7.5, color=MORA, ha='center')
axes[1].annotate('Zona saturada\nf\'≈0', xy=( 3.5, -0.05), fontsize=7.5, color=MORA, ha='center')
axes[1].annotate("f'=1\n(máx)", xy=(0, 1.05), fontsize=8, color=MORA, ha='center')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print('Observaciones clave:')
print('  • Identidad: gradiente siempre = 1 → flujo de gradiente estable, salida ilimitada.')
print('  • Tanh:      gradiente máximo = 1 en z=0, cae a ~0 en los extremos → riesgo de gradiente evanescente.')

---
## Módulo 2 — Regresión: Identidad vs Tanh
Ambas neuronas aprenden `y = mx + c` con datos ruidosos.

> ⚠️ Tanh comprime su salida a (−1, +1). Si `y_real` cae fuera de ese rango, la neurona tanh **nunca** podrá alcanzar los valores reales → MSE estructuralmente alto.

In [ ]:
style = {'description_width': '155px'}
lay   = widgets.Layout(width='420px')

sl_m   = widgets.FloatSlider(value=0.5,  min=-1.0, max=1.0,  step=0.1,  description='Pendiente m:',   style=style, layout=lay)
sl_c   = widgets.FloatSlider(value=0.0,  min=-0.9, max=0.9,  step=0.1,  description='Intercepto c:',  style=style, layout=lay)
sl_sig = widgets.FloatSlider(value=0.1,  min=0.0,  max=0.5,  step=0.05, description='Ruido sigma:',   style=style, layout=lay)
sl_N   = widgets.IntSlider  (value=200,  min=50,   max=500,  step=50,   description='N muestras:',    style=style, layout=lay)
sl_lr  = widgets.FloatLogSlider(value=0.01, base=10, min=-4, max=0, step=0.5, description='Learning rate:', style=style, layout=lay)
sl_ep  = widgets.IntSlider  (value=500,  min=100,  max=3000, step=100,  description='Épocas:',         style=style, layout=lay)
sl_sp  = widgets.FloatSlider(value=0.8,  min=0.5,  max=0.95, step=0.05, description='Train split:',   style=style, layout=lay)

btn_reg = widgets.Button(description='Entrenar ambas neuronas', button_style='success',
                         layout=widgets.Layout(width='230px', height='36px'))
out_reg = widgets.Output()

nota_reg = widgets.HTML(
    '<div style="background:#fff9e6;border-left:4px solid #f0b429;padding:8px 12px;'
    'border-radius:4px;font-size:13px;">'
    '<b>Consejo:</b> Mantén <code>|m|≤1</code> y <code>|c|&lt;1</code> para que '
    '<i>y</i> caiga dentro del rango (−1, +1) de tanh. Así la comparación es justa.</div>'
)

col1 = widgets.VBox([widgets.HTML('<b>Datos</b>'), sl_m, sl_c, sl_sig, sl_N, nota_reg])
col2 = widgets.VBox([widgets.HTML('<b>Entrenamiento</b>'), sl_lr, sl_ep, sl_sp, widgets.HTML('<br>'), btn_reg])
display(widgets.HBox([col1, widgets.HTML('&nbsp;'*6), col2]))
display(out_reg)

def run_reg(b):
    with out_reg:
        clear_output(wait=True)
        np.random.seed(42)
        x   = np.linspace(-3, 3, sl_N.value)
        y   = sl_m.value * x + sl_c.value + np.random.normal(0, sl_sig.value, sl_N.value)
        X   = x.reshape(-1, 1)
        mu, sf = X.mean(), X.std()
        Xn  = (X - mu) / sf
        s   = int(sl_sp.value * sl_N.value)

        n_id   = Neurona(1, activacion='identidad')
        n_tanh = Neurona(1, activacion='tanh')
        n_id.fit(Xn[:s],   y[:s], lr=sl_lr.value, epochs=sl_ep.value)
        n_tanh.fit(Xn[:s], y[:s], lr=sl_lr.value, epochs=sl_ep.value)

        mse_id   = n_id.MSE(n_id.forward(Xn[s:]),   y[s:])
        mse_tanh = n_tanh.MSE(n_tanh.forward(Xn[s:]), y[s:])

        xl  = np.linspace(-3, 3, 300)
        Xln = (xl.reshape(-1,1) - mu) / sf
        y_real   = sl_m.value * xl + sl_c.value
        y_id_p   = n_id.forward(Xln)
        y_tanh_p = n_tanh.forward(Xln)

        fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='white')
        fig.suptitle(
            f'Regresión  |  y = {sl_m.value}x + {sl_c.value}  |  lr={sl_lr.value:.4f}  |  épocas={sl_ep.value}',
            fontsize=12, fontweight='bold'
        )

        for ax, y_p, nombre, color, mse in [
            (axes[0], y_id_p,   'Identidad', AZUL, mse_id),
            (axes[1], y_tanh_p, 'Tanh',      MORA, mse_tanh)
        ]:
            ax.scatter(x[:s], y[:s], color=GRIS, alpha=0.35, s=14, label='Train')
            ax.scatter(x[s:], y[s:], color=ROJO, alpha=0.6,  s=20, marker='^', label='Test')
            ax.plot(xl, y_real, color=VERDE, lw=1.8, ls='--', label='Real')
            ax.plot(xl, y_p,   color=color,  lw=2.5, label=f'{nombre}  MSE={mse:.5f}')
            if nombre == 'Tanh':
                ax.axhline( 1, color='gray', lw=0.8, ls=':', alpha=0.6, label='Límites tanh ±1')
                ax.axhline(-1, color='gray', lw=0.8, ls=':', alpha=0.6)
            ax.set_title(f'Neurona — {nombre}', fontsize=10)
            ax.set_xlabel('x'); ax.set_ylabel('y')
            ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

        axes[2].plot(n_id.loss_hist,   color=AZUL, lw=2, label=f'Identidad  (MSE test={mse_id:.5f})')
        axes[2].plot(n_tanh.loss_hist, color=MORA, lw=2, label=f'Tanh       (MSE test={mse_tanh:.5f})')
        axes[2].set_yscale('log')
        axes[2].set_title('Curvas de pérdida (MSE log)', fontsize=10)
        axes[2].set_xlabel('Época'); axes[2].set_ylabel('MSE')
        axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.25)
        axes[2].spines['top'].set_visible(False); axes[2].spines['right'].set_visible(False)

        plt.tight_layout(); plt.show()

        y_max = abs(y_real).max()
        print(f'Rango real de y: [{y_real.min():.3f}, {y_real.max():.3f}]  |  Rango tanh: (−1, +1)')
        if y_max > 0.95:
            print('⚠️  y supera el rango de tanh → tanh no puede alcanzar los valores reales (ventaja artificial de identidad).')
        print(f'MSE Identidad : {mse_id:.6f}')
        print(f'MSE Tanh      : {mse_tanh:.6f}')
        print(f'=> Mejor en regresión: {"Identidad" if mse_id <= mse_tanh else "Tanh"}')

btn_reg.on_click(run_reg)
run_reg(None)

---
## Módulo 3 — Clasificación binaria: ¿dónde brilla Tanh?

Objetivo: separar dos grupos de puntos 1D con etiquetas **−1** y **+1**.

**¿Cómo funciona la clasificación con una sola neurona?**
- La neurona aprende a producir `ŷ > 0` para la clase +1 y `ŷ < 0` para la clase −1.
- La **frontera de decisión** es el punto x donde `f(z) = 0` → ahí cambia la clase predicha.
- El **accuracy** se calcula como `sign(ŷ) == etiqueta_real`.

> La identidad también puede clasificar correctamente — es un modelo lineal válido.
> La diferencia es que tanh acota la salida a (−1, +1), haciéndola más interpretable como "confianza" en la clase.

In [ ]:
style2 = {'description_width': '155px'}
lay2   = widgets.Layout(width='420px')

sl2_sep = widgets.FloatSlider(value=1.5, min=0.5, max=3.5, step=0.5,  description='Separación clases:', style=style2, layout=lay2)
sl2_sig = widgets.FloatSlider(value=0.6, min=0.1, max=1.5, step=0.1,  description='Ruido sigma:',       style=style2, layout=lay2)
sl2_N   = widgets.IntSlider  (value=300, min=50,  max=600, step=50,   description='N muestras:',        style=style2, layout=lay2)
sl2_lr  = widgets.FloatLogSlider(value=0.05, base=10, min=-4, max=0, step=0.5, description='Learning rate:',  style=style2, layout=lay2)
sl2_ep  = widgets.IntSlider  (value=600, min=100, max=3000, step=100, description='Épocas:',             style=style2, layout=lay2)

btn_cls = widgets.Button(description='Entrenar clasificador', button_style='warning',
                          layout=widgets.Layout(width='230px', height='36px'))
out_cls = widgets.Output()

col1c = widgets.VBox([widgets.HTML('<b>Datos</b>'), sl2_sep, sl2_sig, sl2_N])
col2c = widgets.VBox([widgets.HTML('<b>Entrenamiento</b>'), sl2_lr, sl2_ep, widgets.HTML('<br>'), btn_cls])
display(widgets.HBox([col1c, widgets.HTML('&nbsp;'*6), col2c]))
display(out_cls)

def run_cls(b):
    with out_cls:
        clear_output(wait=True)
        np.random.seed(42)
        sep = sl2_sep.value
        sig = sl2_sig.value
        N   = sl2_N.value

        # ── Generar datos balanceados ──
        n_per_class = N // 2
        x_neg = np.random.normal(-sep, sig, n_per_class)  # clase −1
        x_pos = np.random.normal( sep, sig, n_per_class)  # clase +1
        x_all = np.concatenate([x_neg, x_pos])
        y_all = np.concatenate([np.full(n_per_class, -1.0),
                                np.full(n_per_class,  1.0)])

        # Shuffle manteniendo correspondencia x ↔ y
        idx   = np.random.permutation(len(x_all))
        x_all = x_all[idx]
        y_all = y_all[idx]

        # Normalizar features (controla la escala de z)
        X      = x_all.reshape(-1, 1)
        mu, sf = X.mean(), X.std()
        Xn     = (X - mu) / sf

        s      = int(0.8 * len(x_all))
        X_tr, y_tr = Xn[:s], y_all[:s]
        X_te, y_te = Xn[s:], y_all[s:]

        # ── Entrenar ──
        n_id   = Neurona(1, activacion='identidad')
        n_tanh = Neurona(1, activacion='tanh')
        n_id.fit(X_tr,   y_tr, lr=sl2_lr.value, epochs=sl2_ep.value)
        n_tanh.fit(X_tr, y_tr, lr=sl2_lr.value, epochs=sl2_ep.value)

        # ── Accuracy: umbral natural = 0  →  clase = sign(ŷ) ──
        # La neurona aprende w y b tal que z=0 coincide con la frontera entre clases.
        # sign(f(z)) es equivalente a sign(z) para ambas funciones (ambas son monótonas).
        raw_id   = n_id.forward(X_te)
        raw_tanh = n_tanh.forward(X_te)

        pred_id   = np.where(raw_id   >= 0, 1.0, -1.0)
        pred_tanh = np.where(raw_tanh >= 0, 1.0, -1.0)

        acc_id   = np.mean(pred_id   == y_te) * 100
        acc_tanh = np.mean(pred_tanh == y_te) * 100

        # ── Curvas de salida en espacio original ──
        xl   = np.linspace(x_all.min() - 0.5, x_all.max() + 0.5, 500)
        Xln  = (xl.reshape(-1, 1) - mu) / sf
        y_id_p   = n_id.forward(Xln)
        y_tanh_p = n_tanh.forward(Xln)

        # Frontera de decisión: x donde f(z)=0 (cruce de signo)
        def frontera(y_curve, xl):
            cruces = np.where(np.diff(np.sign(y_curve)))[0]
            if len(cruces) == 0: return None
            i  = cruces[0]
            x0 = xl[i] - y_curve[i] * (xl[i+1] - xl[i]) / (y_curve[i+1] - y_curve[i])
            return x0

        front_id   = frontera(y_id_p,   xl)
        front_tanh = frontera(y_tanh_p, xl)

        # ── Figura ──
        fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='white')
        fig.suptitle(
            f'Clasificación  |  sep={sep}  σ={sig}  |  lr={sl2_lr.value:.4f}  |  épocas={sl2_ep.value}',
            fontsize=12, fontweight='bold'
        )

        for ax, y_p, nombre, color, acc, front in [
            (axes[0], y_id_p,   'Identidad', AZUL, acc_id,   front_id),
            (axes[1], y_tanh_p, 'Tanh',      MORA, acc_tanh, front_tanh)
        ]:
            # Puntos en su posición x real, altura = etiqueta real (−1 o +1)
            mask_neg = (y_all == -1)
            mask_pos = (y_all ==  1)
            ax.scatter(x_all[mask_neg], y_all[mask_neg],
                       color=AZUL, alpha=0.4, s=18, zorder=3, label='Clase −1 (etiq.)')
            ax.scatter(x_all[mask_pos], y_all[mask_pos],
                       color=ROJO, alpha=0.4, s=18, zorder=3, label='Clase +1 (etiq.)')

            # Curva continua de salida f(z)
            ax.plot(xl, y_p, color=color, lw=2.5, zorder=4,
                    label=f'{nombre}  acc={acc:.1f}%')
            ax.axhline(0, color='gray', lw=1.0, ls='--', alpha=0.8, label='Umbral = 0')

            # Sombreado de zonas de decisión
            ax.fill_between(xl, -2, 0,    alpha=0.04, color=AZUL)
            ax.fill_between(xl,  0,  2,   alpha=0.04, color=ROJO)

            # Línea vertical: frontera de decisión
            if front is not None:
                ax.axvline(front, color=color, lw=1.8, ls=':', alpha=0.9,
                           label=f'Frontera x≈{front:.2f}')

            if nombre == 'Tanh':
                ax.axhline( 1, color='lightgray', lw=0.8, ls=':')
                ax.axhline(-1, color='lightgray', lw=0.8, ls=':')

            ax.set_title(f'Neurona — {nombre}\nAcc test = {acc:.1f}%', fontsize=10)
            ax.set_xlabel('x  (espacio original)')
            ax.set_ylabel('salida f(z)')
            ax.set_ylim(-1.8, 1.8)
            ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

        # Curvas de pérdida
        axes[2].plot(n_id.loss_hist,   color=AZUL, lw=2, label=f'Identidad  acc={acc_id:.1f}%')
        axes[2].plot(n_tanh.loss_hist, color=MORA, lw=2, label=f'Tanh       acc={acc_tanh:.1f}%')
        axes[2].set_yscale('log')
        axes[2].set_title('Curvas de pérdida (MSE log)', fontsize=10)
        axes[2].set_xlabel('Época'); axes[2].set_ylabel('MSE')
        axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.25)
        axes[2].spines['top'].set_visible(False); axes[2].spines['right'].set_visible(False)

        plt.tight_layout(); plt.show()

        print(f'Accuracy Identidad : {acc_id:.1f}%  (umbral = 0, predicción = sign(z))')
        print(f'Accuracy Tanh      : {acc_tanh:.1f}%  (umbral = 0, predicción = sign(tanh(z)))')
        print()
        print('Nota: sign(f(z)) = sign(z) para cualquier función monótona que pase por 0.')
        print('Ambos modelos son lineales en z → misma frontera de decisión (hiperplano), distinta forma de salida.')

btn_cls.on_click(run_cls)
run_cls(None)

---
## Módulo 4 — Desvanecimiento de Gradiente (Vanishing Gradient)

El **desvanecimiento de gradiente** ocurre cuando la derivada de la función de activación se acerca a 0,
haciendo que el gradiente se multiplique por casi cero en cada capa → el aprendizaje se frena o detiene.

```
∂L/∂w = (∂L/∂ŷ) · f'(z) · x
         ──────    ──────
         OK        ≈ 0 si tanh está saturada  →  ∂L/∂w ≈ 0
```

**¿Por qué afecta a Tanh?**  
`f'(z) = 1 − tanh²(z) → 0` cuando `|z| >> 0` (zonas de saturación).

**¿Por qué NO afecta a Identidad?**  
`f'(z) = 1` siempre → el gradiente nunca se encoge.

> **Prueba:** aumenta **Escala de entrada** para saturar la tanh y observa cómo su gradiente colapsa.

In [ ]:
style3 = {'description_width': '165px'}
lay3   = widgets.Layout(width='440px')

sl3_z   = widgets.FloatSlider(value=1.0, min=0.5, max=8.0, step=0.5,
    description='Escala de entrada:', style=style3, layout=lay3)
sl3_ep  = widgets.IntSlider  (value=800, min=100, max=3000, step=100,
    description='Épocas:', style=style3, layout=lay3)
sl3_lr  = widgets.FloatLogSlider(value=0.05, base=10, min=-4, max=0, step=0.5,
    description='Learning rate:', style=style3, layout=lay3)
sl3_N   = widgets.IntSlider  (value=200, min=50, max=500, step=50,
    description='N muestras:', style=style3, layout=lay3)

btn_vg = widgets.Button(description='Simular desvanecimiento', button_style='danger',
                         layout=widgets.Layout(width='240px', height='36px'))
out_vg = widgets.Output()

nota_vg = widgets.HTML(
    '<div style="background:#fef0f0;border-left:4px solid #D85A30;padding:8px 12px;'
    'border-radius:4px;font-size:13px;">'
    '<b>Prueba:</b> Escala ≤ 2 → sin saturación, gradientes similares. '
    'Escala ≥ 4 → tanh saturada, gradiente colapsa a casi 0. '
    'Observa el panel <i>Magnitud del gradiente</i> y la <i>Velocidad de aprendizaje</i>.</div>'
)

col1v = widgets.VBox([widgets.HTML('<b>Configuración</b>'), sl3_z, sl3_ep, sl3_lr, sl3_N, nota_vg])
col2v = widgets.VBox([widgets.HTML('<br>'), btn_vg])
display(widgets.HBox([col1v, widgets.HTML('&nbsp;'*6), col2v]))
display(out_vg)

def run_vg(b):
    with out_vg:
        clear_output(wait=True)
        np.random.seed(42)
        z_scale = sl3_z.value
        N       = sl3_N.value
        epochs  = sl3_ep.value
        lr      = sl3_lr.value

        # Datos con escala grande (NO normalizamos → z grande → satura tanh)
        x_raw = np.random.uniform(-1, 1, N) * z_scale
        y_raw = 0.5 * x_raw + np.random.normal(0, 0.05, N)
        X     = x_raw.reshape(-1, 1)
        s     = int(0.8 * N)

        n_id   = Neurona(1, activacion='identidad')
        n_tanh = Neurona(1, activacion='tanh')
        n_id.fit(X[:s],   y_raw[:s], lr=lr, epochs=epochs)
        n_tanh.fit(X[:s], y_raw[:s], lr=lr, epochs=epochs)

        # Derivada promedio en el rango de los datos
        z_demo          = np.linspace(-z_scale, z_scale, 500)
        grad_id_demo    = df_identidad(z_demo)  # = 1 siempre
        grad_tanh_demo  = df_tanh(z_demo)
        mean_grad_tanh  = float(grad_tanh_demo.mean())

        # Valores finales de gradiente
        g_fin_id   = n_id.grad_w_hist[-1]
        g_fin_tanh = n_tanh.grad_w_hist[-1]
        ratio      = g_fin_tanh / (g_fin_id + 1e-12)

        # ── Figura: 4 paneles ──
        fig = plt.figure(figsize=(16, 10), facecolor='white')
        fig.suptitle(
            f'Desvanecimiento de Gradiente  |  escala={z_scale}  |  lr={lr:.4f}  |  épocas={epochs}',
            fontsize=12, fontweight='bold'
        )
        gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
        ax1 = fig.add_subplot(gs[0, 0])   # f'(z) según rango de datos
        ax2 = fig.add_subplot(gs[0, 1])   # |∇w| a lo largo del entrenamiento
        ax3 = fig.add_subplot(gs[1, 0])   # curvas de pérdida
        ax4 = fig.add_subplot(gs[1, 1])   # velocidad de reducción del error

        # Panel 1: Derivada f'(z) en el rango real de entrada
        ax1.plot(z_demo, grad_id_demo,   color=AZUL, lw=2.5, label="f'(z) Identidad = 1")
        ax1.plot(z_demo, grad_tanh_demo, color=MORA, lw=2.5, label="f'(z) Tanh")
        ax1.fill_between(z_demo, grad_tanh_demo, alpha=0.15, color=MORA)
        ax1.axvspan(-z_scale, z_scale, alpha=0.07, color=ROJO,
                    label=f'Rango de x (±{z_scale})')
        ax1.set_title("Derivada f'(z) en el rango de los datos", fontsize=10)
        ax1.set_xlabel('z'); ax1.set_ylabel("f'(z)")
        ax1.set_ylim(-0.1, 1.3)
        ax1.legend(fontsize=8); ax1.grid(True, alpha=0.25)
        ax1.text(0, 1.2,
                 f"f' promedio de tanh en ±{z_scale}: {mean_grad_tanh:.4f}",
                 fontsize=8.5, color=MORA, ha='center',
                 bbox=dict(boxstyle='round', facecolor='#f3eaff', alpha=0.85))
        ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

        # Panel 2: Magnitud del gradiente |∂L/∂w| a lo largo del entrenamiento
        ax2.plot(n_id.grad_w_hist,   color=AZUL, lw=2, label='|∇w| Identidad')
        ax2.plot(n_tanh.grad_w_hist, color=MORA, lw=2, label='|∇w| Tanh')
        ax2.set_yscale('log')
        ax2.set_title('Magnitud del gradiente |∂L/∂w| por época', fontsize=10)
        ax2.set_xlabel('Época'); ax2.set_ylabel('|∂L/∂w|  (escala log)')
        ax2.legend(fontsize=9); ax2.grid(True, alpha=0.25)
        ax2.text(0.97, 0.97,
                 f'Ratio final\ntanh / identidad = {ratio:.5f}',
                 transform=ax2.transAxes, ha='right', va='top', fontsize=9,
                 bbox=dict(boxstyle='round', facecolor='#fff0f0', alpha=0.9))
        ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

        # Panel 3: Curvas de pérdida (MSE)
        ax3.plot(n_id.loss_hist,   color=AZUL, lw=2,
                 label=f'Identidad  (final={n_id.loss_hist[-1]:.5f})')
        ax3.plot(n_tanh.loss_hist, color=MORA, lw=2,
                 label=f'Tanh       (final={n_tanh.loss_hist[-1]:.5f})')
        ax3.set_yscale('log')
        ax3.set_title('Curvas de pérdida (MSE log)', fontsize=10)
        ax3.set_xlabel('Época'); ax3.set_ylabel('MSE')
        ax3.legend(fontsize=9); ax3.grid(True, alpha=0.25)
        ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

        # Panel 4: Velocidad de aprendizaje (caída de MSE entre épocas)
        delta_id   = np.clip(-np.diff(n_id.loss_hist),   0, None)
        delta_tanh = np.clip(-np.diff(n_tanh.loss_hist), 0, None)
        w_size = max(1, epochs // 40)
        smooth = lambda a: np.convolve(a, np.ones(w_size) / w_size, mode='valid')
        ax4.plot(smooth(delta_id),   color=AZUL, lw=2, label='Identidad')
        ax4.plot(smooth(delta_tanh), color=MORA, lw=2, label='Tanh')
        ax4.set_title('Velocidad de reducción del error\n(ΔMSE por época, suavizado)', fontsize=10)
        ax4.set_xlabel('Época'); ax4.set_ylabel('ΔMSE')
        ax4.legend(fontsize=9); ax4.grid(True, alpha=0.25)
        ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)

        plt.tight_layout(); plt.show()

        print(f'Escala de entrada: ±{z_scale}')
        print(f"f' promedio de tanh en ese rango : {mean_grad_tanh:.6f}")
        print(f"f' de identidad en ese rango     : 1.000000  (siempre)")
        print()
        print(f'|∇w| final  Identidad : {g_fin_id:.4e}')
        print(f'|∇w| final  Tanh      : {g_fin_tanh:.4e}')
        print(f'Ratio tanh / identidad al final : {ratio:.6f}')
        print()
        if ratio < 0.05:
            print('🔴 DESVANECIMIENTO SEVERO: el gradiente de tanh es < 5% del de identidad.')
            print('   La tanh está muy saturada → aprendizaje prácticamente detenido.')
        elif ratio < 0.3:
            print(f'🟡 Desvanecimiento notable: gradiente de tanh es {ratio*100:.1f}% del de identidad.')
        else:
            print('🟢 Poco desvanecimiento. Aumenta la escala de entrada (≥ 4) para verlo claramente.')

btn_vg.on_click(run_vg)
run_vg(None)

---
## Resumen conceptual

| Concepto | Descripción |
|---|---|
| **Identidad** | `f(z) = z`, derivada = 1 siempre. Sin saturación. Ideal para **regresión** |
| **Tanh** | `f(z) = tanh(z)`, rango (−1,+1). Derivada `1−tanh²(z)`, ≈0 en saturación. Ideal para **clasificación** o capas ocultas |
| **Clasificación con neurona lineal** | Ambas funciones aprenden la frontera correcta. El accuracy se calcula con `sign(ŷ)`, umbral = 0. La identidad también clasifica bien |
| **Desvanecimiento de gradiente** | Cuando `|z|` es grande, `f'(z)≈0` en tanh → gradiente ≈ 0 → aprendizaje frenado. Identidad es inmune (f'=1) |
| **Normalización** | Mantener z en rango pequeño (|z|≤2) reduce la saturación de tanh y el desvanecimiento |
| **¿Cuándo usar cuál?** | Regresión / salida ilimitada → identidad. Clasificación / capa oculta → tanh (o ReLU que evita saturación) |